# 📊 Birmarket.az Inflation Analysis

This notebook compares product prices between two months to measure inflation/deflation trends.

## Metrics Calculated:
- **Price Change (%)**: Percentage change in prices
- **Inflation Rate**: Overall weighted average price change
- **Category-wise Analysis**: Price changes by product category
- **Top Gainers/Losers**: Products with highest price changes

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuration
SNAPSHOTS_DIR = '../data/snapshots'
REPORTS_DIR = '../reports'
SOURCE = 'bazarstore' # Options: 'arazmarket', 'bazarstore', 'price_history' (for legacy Birmarket)

print("📁 Available monthly snapshots:")
snapshots = sorted([f for f in os.listdir(SNAPSHOTS_DIR) if f.endswith('.csv')])
for f in snapshots:
    print(f"   - {f}")

📁 Available monthly snapshots:
   - price_history_202601.csv
   - price_history_202602.csv


In [ ]:
# Select months for comparison
# Format: YYYYMM (e.g., 202601 for January 2026)

PREVIOUS_MONTH = '202601'  # Baseline month
CURRENT_MONTH = '202602'   # Current month to compare

print(f"📊 Analyzing: {SOURCE}")
print(f"📅 Comparing: {PREVIOUS_MONTH} → {CURRENT_MONTH}")

📅 Comparing: 202601 → 202602


In [ ]:
# Load the data
# Adjust filename format if needed (legacy Birmarket used 'price_history_YYYYMM.csv')
if SOURCE == 'price_history':
    prev_file = f"{SNAPSHOTS_DIR}/{SOURCE}_{PREVIOUS_MONTH}.csv"
    curr_file = f"{SNAPSHOTS_DIR}/{SOURCE}_{CURRENT_MONTH}.csv"
else:
    prev_file = f"{SNAPSHOTS_DIR}/{SOURCE}_{PREVIOUS_MONTH}.csv"
    curr_file = f"{SNAPSHOTS_DIR}/{SOURCE}_{CURRENT_MONTH}.csv"

# Note: If files are missing, check if you need to rename them to match naming convention
print(f"Loading {PREVIOUS_MONTH} data from {prev_file}...")
df_prev = pd.read_csv(prev_file)
print(f"  ✓ {len(df_prev):,} records loaded")

print(f"Loading {CURRENT_MONTH} data from {curr_file}...")
df_curr = pd.read_csv(curr_file)
print(f"  ✓ {len(df_curr):,} records loaded")

# Add month identifiers
df_prev['month'] = PREVIOUS_MONTH
df_curr['month'] = CURRENT_MONTH

Loading 202601 data...
  ✓ 6,226 records loaded
Loading 202602 data...
  ✓ 590,102 records loaded


In [4]:
# Preview data structure
print("📋 Data Structure:")
print(df_curr.dtypes)
print("\n📊 Sample Data:")
df_curr.head(3)

📋 Data Structure:
id                    int64
name                 object
brand                object
category             object
category_id           int64
price               float64
base_price          float64
discount_percent    float64
rating              float64
rating_count          int64
seller_rating         int64
is_new                 bool
currency             object
url                  object
image                object
timestamp            object
month                object
dtype: object

📊 Sample Data:


,id,name,brand,category,category_id,price,base_price,discount_percent,rating,rating_count,seller_rating,is_new,currency,url,image,timestamp,month
0,326243,Paltaryuyan maşın Hoffmann VSL-6100S,Hoffmann,Paltaryuyan maşınlar,109,477.98,800.0,40.25,4.3,46,99,False,AZN,https://birmarket.az/product/326243-paltaryuya...,https://strgimgr.umico.az/sized/280/326243-db0...,2026-02-03 08:45:55,202602
1,1025158,Elektrik çaydan Engy KEP0302-GS,Engy,Elektrik çaydanlar və termopotlar,140,10.90,39.0,72.05,4.7,16,95,False,AZN,https://birmarket.az/product/1025158-elektrik-...,https://strgimgr.umico.az/img/product/280/03f0...,2026-02-03 08:45:55,202602
2,1124405,Soyuducu Beko B1RCNK272G,Beko,Soyuducular,108,654.00,1043.0,37.30,4.8,19,95,False,AZN,https://birmarket.az/product/1124405-soyuducu-...,https://strgimgr.umico.az/img/product/280/c010...,2026-02-03 08:45:55,202602


## 🔄 Data Preparation

Aggregate data by product ID to handle potential duplicates and prepare for comparison.

In [5]:
def aggregate_monthly_data(df):
    """Aggregate data by product ID, taking average prices."""
    return df.groupby('id').agg({
        'name': 'first',
        'brand': 'first',
        'category': 'first',
        'category_id': 'first',
        'price': 'mean',
        'base_price': 'mean',
        'discount_percent': 'mean',
        'month': 'first'
    }).reset_index()

prev_agg = aggregate_monthly_data(df_prev)
curr_agg = aggregate_monthly_data(df_curr)

print(f"📊 Unique products in {PREVIOUS_MONTH}: {len(prev_agg):,}")
print(f"📊 Unique products in {CURRENT_MONTH}: {len(curr_agg):,}")

📊 Unique products in 202601: 6,226
📊 Unique products in 202602: 585,457


In [6]:
# Merge datasets for comparison
comparison = pd.merge(
    prev_agg[['id', 'name', 'brand', 'category', 'category_id', 'price', 'base_price']],
    curr_agg[['id', 'price', 'base_price', 'discount_percent']],
    on='id',
    how='inner',
    suffixes=('_prev', '_curr')
)

print(f"🔗 Products matched in both months: {len(comparison):,}")
print(f"   Products only in {PREVIOUS_MONTH}: {len(prev_agg) - len(comparison):,}")
print(f"   Products only in {CURRENT_MONTH} (new): {len(curr_agg) - len(comparison):,}")

🔗 Products matched in both months: 2
   Products only in 202601: 6,224
   Products only in 202602 (new): 585,455


## 📈 Price Change Calculations

In [7]:
# Calculate price changes
comparison['price_change'] = comparison['price_curr'] - comparison['price_prev']
comparison['price_change_pct'] = ((comparison['price_curr'] - comparison['price_prev']) / comparison['price_prev']) * 100

# Handle edge cases (division by zero)
comparison['price_change_pct'] = comparison['price_change_pct'].replace([np.inf, -np.inf], np.nan)
comparison = comparison.dropna(subset=['price_change_pct'])

# Classify changes
comparison['change_type'] = pd.cut(
    comparison['price_change_pct'],
    bins=[-np.inf, -10, -1, 1, 10, np.inf],
    labels=['Big Decrease (>10%)', 'Small Decrease (1-10%)', 'Stable', 'Small Increase (1-10%)', 'Big Increase (>10%)']
)

print("✅ Price changes calculated!")
comparison[['name', 'price_prev', 'price_curr', 'price_change', 'price_change_pct']].head()

✅ Price changes calculated!


,name,price_prev,price_curr,price_change,price_change_pct
0,"Soyuq meyvə çayı Zhenchun Naringi dadı ilə, 49...",10.2,10.2,0.0,0.0
1,Soyuq meyvə çayı Zhenchun albalı və çiyələk da...,10.2,10.2,0.0,0.0


## 📊 Overall Inflation Summary

In [8]:
# Calculate overall inflation metrics
total_prev = comparison['price_prev'].sum()
total_curr = comparison['price_curr'].sum()
overall_inflation = ((total_curr - total_prev) / total_prev) * 100

avg_price_prev = comparison['price_prev'].mean()
avg_price_curr = comparison['price_curr'].mean()
median_change = comparison['price_change_pct'].median()

print("=" * 60)
print("📊 OVERALL INFLATION SUMMARY")
print("=" * 60)
print(f"\n📅 Period: {PREVIOUS_MONTH} → {CURRENT_MONTH}")
print(f"📦 Products Compared: {len(comparison):,}")
print(f"\n💰 Average Price ({PREVIOUS_MONTH}): {avg_price_prev:.2f} AZN")
print(f"💰 Average Price ({CURRENT_MONTH}): {avg_price_curr:.2f} AZN")
print(f"\n📈 OVERALL INFLATION RATE: {overall_inflation:+.2f}%")
print(f"📊 Median Price Change: {median_change:+.2f}%")
print("=" * 60)

📊 OVERALL INFLATION SUMMARY

📅 Period: 202601 → 202602
📦 Products Compared: 2

💰 Average Price (202601): 10.20 AZN
💰 Average Price (202602): 10.20 AZN

📈 OVERALL INFLATION RATE: +0.00%
📊 Median Price Change: +0.00%


In [9]:
# Distribution of price changes
print("\n📊 PRICE CHANGE DISTRIBUTION")
print("-" * 40)
distribution = comparison['change_type'].value_counts()
for change_type, count in distribution.items():
    pct = (count / len(comparison)) * 100
    print(f"{change_type}: {count:,} products ({pct:.1f}%)")

# Summary stats
increased = (comparison['price_change_pct'] > 1).sum()
decreased = (comparison['price_change_pct'] < -1).sum()
stable = len(comparison) - increased - decreased

print(f"\n📈 Prices Increased: {increased:,} ({increased/len(comparison)*100:.1f}%)")
print(f"📉 Prices Decreased: {decreased:,} ({decreased/len(comparison)*100:.1f}%)")
print(f"➡️  Prices Stable: {stable:,} ({stable/len(comparison)*100:.1f}%)")


📊 PRICE CHANGE DISTRIBUTION
----------------------------------------
Stable: 2 products (100.0%)
Big Decrease (>10%): 0 products (0.0%)
Small Decrease (1-10%): 0 products (0.0%)
Small Increase (1-10%): 0 products (0.0%)
Big Increase (>10%): 0 products (0.0%)

📈 Prices Increased: 0 (0.0%)
📉 Prices Decreased: 0 (0.0%)
➡️  Prices Stable: 2 (100.0%)


## 🏷️ Category-wise Inflation Analysis

In [10]:
# Calculate inflation by category
category_stats = comparison.groupby('category').agg({
    'id': 'count',
    'price_prev': 'sum',
    'price_curr': 'sum',
    'price_change_pct': ['mean', 'median', 'std']
}).round(2)

category_stats.columns = ['product_count', 'total_prev', 'total_curr', 'avg_change_pct', 'median_change_pct', 'std_change_pct']
category_stats['category_inflation'] = ((category_stats['total_curr'] - category_stats['total_prev']) / category_stats['total_prev'] * 100).round(2)

# Sort by inflation rate
category_stats = category_stats.sort_values('category_inflation', ascending=False)

print("\n📊 INFLATION BY CATEGORY")
print("=" * 80)
category_stats[['product_count', 'category_inflation', 'avg_change_pct', 'median_change_pct']]


📊 INFLATION BY CATEGORY


,product_count,category_inflation,avg_change_pct,median_change_pct
category,,,,
Soyuq çay və kofelər,2,0.0,0.0,0.0


In [ ]:
# Top 10 categories with highest inflation
print("\n🔥 TOP 10 CATEGORIES WITH HIGHEST INFLATION")
print("-" * 60)
top_inflation = category_stats.head(10)
for idx, (cat, row) in enumerate(top_inflation.iterrows(), 1):
    print(f"{idx:2}. {cat[:40]:<40} {row['category_inflation']:+7.2f}% ({int(row['product_count'])} products)")

print("\n❄️ TOP 10 CATEGORIES WITH LOWEST INFLATION (DEFLATION)")
print("-" * 60)
bottom_inflation = category_stats.tail(10).iloc[::-1]
for idx, (cat, row) in enumerate(bottom_inflation.iterrows(), 1):
    print(f"{idx:2}. {cat[:40]:<40} {row['category_inflation']:+7.2f}% ({int(row['product_count'])} products)")

: 

: 

: 

## 🏆 Top Price Changers

In [ ]:
# Top products with highest price increase
top_increases = comparison.nlargest(20, 'price_change_pct')[['name', 'brand', 'category', 'price_prev', 'price_curr', 'price_change', 'price_change_pct']]

print("\n📈 TOP 20 PRODUCTS WITH HIGHEST PRICE INCREASE")
print("=" * 100)
for idx, row in top_increases.iterrows():
    print(f"• {row['name'][:50]:<50}")
    print(f"  {row['price_prev']:.2f} → {row['price_curr']:.2f} AZN ({row['price_change_pct']:+.1f}%)")
    print()

: 

: 

: 

In [ ]:
# Top products with highest price decrease
top_decreases = comparison.nsmallest(20, 'price_change_pct')[['name', 'brand', 'category', 'price_prev', 'price_curr', 'price_change', 'price_change_pct']]

print("\n📉 TOP 20 PRODUCTS WITH HIGHEST PRICE DECREASE")
print("=" * 100)
for idx, row in top_decreases.iterrows():
    print(f"• {row['name'][:50]:<50}")
    print(f"  {row['price_prev']:.2f} → {row['price_curr']:.2f} AZN ({row['price_change_pct']:+.1f}%)")
    print()

: 

: 

: 

In [ ]:
# Largest absolute price changes (AZN)
comparison['abs_change'] = comparison['price_change'].abs()
largest_changes = comparison.nlargest(20, 'abs_change')[['name', 'brand', 'category', 'price_prev', 'price_curr', 'price_change', 'price_change_pct']]

print("\n💰 TOP 20 LARGEST ABSOLUTE PRICE CHANGES (AZN)")
print("=" * 100)
for idx, row in largest_changes.iterrows():
    direction = "📈" if row['price_change'] > 0 else "📉"
    print(f"{direction} {row['name'][:45]:<45}")
    print(f"   {row['price_prev']:.2f} → {row['price_curr']:.2f} AZN (Change: {row['price_change']:+.2f} AZN / {row['price_change_pct']:+.1f}%)")
    print()

: 

: 

: 

## 🏷️ Brand Analysis

In [ ]:
# Inflation by brand (for brands with 5+ products)
brand_stats = comparison.groupby('brand').agg({
    'id': 'count',
    'price_prev': 'sum',
    'price_curr': 'sum',
    'price_change_pct': 'mean'
}).round(2)

brand_stats.columns = ['product_count', 'total_prev', 'total_curr', 'avg_change_pct']
brand_stats['brand_inflation'] = ((brand_stats['total_curr'] - brand_stats['total_prev']) / brand_stats['total_prev'] * 100).round(2)

# Filter brands with significant product count
significant_brands = brand_stats[brand_stats['product_count'] >= 5].sort_values('brand_inflation', ascending=False)

print(f"\n🏷️ BRAND INFLATION ANALYSIS (Brands with 5+ products)")
print(f"Total significant brands: {len(significant_brands)}")
print("\n🔥 Top 15 Brands with Highest Inflation:")
print("-" * 60)
for idx, (brand, row) in enumerate(significant_brands.head(15).iterrows(), 1):
    print(f"{idx:2}. {brand[:35]:<35} {row['brand_inflation']:+7.2f}% ({int(row['product_count'])} products)")

: 

: 

: 

## 📋 Export Results

In [ ]:
# Ensure reports directory exists
os.makedirs(REPORTS_DIR, exist_ok=True)

# Generate report filename
report_date = datetime.now().strftime('%Y%m%d')
report_prefix = f"{REPORTS_DIR}/inflation_{PREVIOUS_MONTH}_vs_{CURRENT_MONTH}"

# Export detailed comparison
comparison_export = comparison[['id', 'name', 'brand', 'category', 'price_prev', 'price_curr', 'price_change', 'price_change_pct', 'change_type']].copy()
comparison_export.columns = ['Product ID', 'Name', 'Brand', 'Category', f'Price {PREVIOUS_MONTH}', f'Price {CURRENT_MONTH}', 'Price Change (AZN)', 'Price Change (%)', 'Change Type']
comparison_export.to_csv(f"{report_prefix}_detailed.csv", index=False)
comparison_export.to_excel(f"{report_prefix}_detailed.xlsx", index=False)

# Export category summary
category_stats.to_csv(f"{report_prefix}_by_category.csv")
category_stats.to_excel(f"{report_prefix}_by_category.xlsx")

# Export brand summary
significant_brands.to_csv(f"{report_prefix}_by_brand.csv")
significant_brands.to_excel(f"{report_prefix}_by_brand.xlsx")

print(f"\n✅ Reports exported to {REPORTS_DIR}/")
print(f"   - {report_prefix}_detailed.csv")
print(f"   - {report_prefix}_detailed.xlsx")
print(f"   - {report_prefix}_by_category.csv")
print(f"   - {report_prefix}_by_category.xlsx")
print(f"   - {report_prefix}_by_brand.csv")
print(f"   - {report_prefix}_by_brand.xlsx")

: 

: 

: 

## 📊 Summary Statistics

In [ ]:
# Final summary
print("\n" + "=" * 70)
print("📊 INFLATION ANALYSIS SUMMARY")
print("=" * 70)
print(f"\n📅 Comparison Period: {PREVIOUS_MONTH} → {CURRENT_MONTH}")
print(f"\n📦 Data Overview:")
print(f"   • Products in {PREVIOUS_MONTH}: {len(prev_agg):,}")
print(f"   • Products in {CURRENT_MONTH}: {len(curr_agg):,}")
print(f"   • Products matched for comparison: {len(comparison):,}")
print(f"\n💰 Price Statistics:")
print(f"   • Average price ({PREVIOUS_MONTH}): {avg_price_prev:.2f} AZN")
print(f"   • Average price ({CURRENT_MONTH}): {avg_price_curr:.2f} AZN")
print(f"   • Overall Inflation Rate: {overall_inflation:+.2f}%")
print(f"   • Median Price Change: {median_change:+.2f}%")
print(f"\n📈 Price Movement:")
print(f"   • Products with price increase: {increased:,} ({increased/len(comparison)*100:.1f}%)")
print(f"   • Products with price decrease: {decreased:,} ({decreased/len(comparison)*100:.1f}%)")
print(f"   • Products with stable price: {stable:,} ({stable/len(comparison)*100:.1f}%)")
print(f"\n🏷️ Category Insights:")
print(f"   • Highest inflation category: {category_stats.index[0]} ({category_stats.iloc[0]['category_inflation']:+.2f}%)")
print(f"   • Lowest inflation category: {category_stats.index[-1]} ({category_stats.iloc[-1]['category_inflation']:+.2f}%)")
print("\n" + "=" * 70)

: 

: 

: 